# Part 07 — RAG: Retrieval-Augmented Generation

**Core idea:** LLMs have a fixed knowledge cutoff and no access to private data. RAG solves this by fetching relevant documents at query time and injecting them into the prompt — grounding the answer in real sources rather than memorized statistics.

```
┌──────────────────── RAG Pipeline ──────────────────────┐
│                                                         │
│  Query ──► Embed Query ──► Search Vector DB             │
│                                 │                       │
│                           Top-K Chunks                  │
│                                 │                       │
│  Question + Context ──────────► LLM ──► Answer          │
└─────────────────────────────────────────────────────────┘
```

### What this notebook covers
| Section | Key concept |
|---------|------------|
| Why RAG? | 4 problems RAG solves |
| Model types | Auto-regressive vs auto-encoding — which to use where |
| Vector embeddings | How meaning becomes searchable numbers |
| Document loading | Reading txt, PDF into Python |
| Text chunking | Splitting docs with overlap |
| ChromaDB | Building and querying a vector store |
| Similarity search | MMR vs cosine similarity |
| LCEL chain | End-to-end LangChain RAG pipeline |
| Keyword retrieval | BM25-style search without embeddings |
| Gradio UI | Wrapping RAG in a web interface |
| Evaluation | LLM-as-judge scoring |

---
## Why RAG?

| Problem | Without RAG | With RAG |
|---------|------------|----------|
| **Private data** | LLM has never seen it | Retrieved at query time |
| **Hallucination** | LLM fabricates plausible-sounding facts | Answer grounded in real chunks |
| **Knowledge cutoff** | Frozen at training date | Always reflects current docs |
| **Context limit** | Everything must fit in one prompt | Only relevant chunks injected |

**When NOT to use RAG:**
- Question is answered by model's training knowledge (no private data needed)
- Latency is critical (RAG adds a retrieval round-trip)
- Docs are too long and you need cross-document reasoning — consider fine-tuning instead

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.patches import FancyArrowPatch

fig, ax = plt.subplots(figsize=(14, 5))
ax.set_xlim(0, 14); ax.set_ylim(0, 5); ax.axis('off')
fig.patch.set_facecolor('#f8f9fa')

# ── Indexing pipeline (top row) ───────────────────────────────────────────
index_steps = [
    (1.0, 3.8, "Documents\n(PDF/txt)", "#3498db"),
    (3.5, 3.8, "Chunking\n(split + overlap)", "#9b59b6"),
    (6.0, 3.8, "Embedding\nModel", "#e67e22"),
    (8.5, 3.8, "Vector DB\n(ChromaDB)", "#27ae60"),
]
# ── Query pipeline (bottom row) ───────────────────────────────────────────
query_steps = [
    (1.0, 1.5, "User\nQuery", "#e74c3c"),
    (3.5, 1.5, "Embed\nQuery", "#e67e22"),
    (6.0, 1.5, "Similarity\nSearch", "#27ae60"),
    (8.5, 1.5, "Top-K\nChunks", "#2ecc71"),
    (11.0, 1.5, "LLM +\nContext", "#8e44ad"),
    (13.0, 1.5, "Answer", "#c0392b"),
]

def draw_box(ax, x, y, label, color, width=1.8, height=0.9):
    rect = plt.Rectangle((x - width/2, y - height/2), width, height,
                          color=color, alpha=0.85, zorder=3, linewidth=1.5,
                          edgecolor='white')
    ax.add_patch(rect)
    ax.text(x, y, label, ha='center', va='center', fontsize=8.5,
            color='white', fontweight='bold', zorder=4)

def draw_arrow(ax, x1, x2, y, color='#555', linestyle='-'):
    ax.annotate("", xy=(x2 - 0.9, y), xytext=(x1 + 0.9, y),
                arrowprops=dict(arrowstyle="-|>", color=color,
                                lw=1.5, linestyle=linestyle))

# Draw indexing row
for x, y, label, color in index_steps:
    draw_box(ax, x, y, label, color)

# Arrows between indexing steps
for i in range(len(index_steps) - 1):
    draw_arrow(ax, index_steps[i][0], index_steps[i+1][0], 3.8)

# Draw query row
for x, y, label, color in query_steps:
    draw_box(ax, x, y, label, color)

# Arrows between query steps
for i in range(len(query_steps) - 1):
    draw_arrow(ax, query_steps[i][0], query_steps[i+1][0], 1.5)

# Vertical arrow: Vector DB → Top-K Chunks
ax.annotate("", xy=(8.5, 2.0), xytext=(8.5, 3.35),
            arrowprops=dict(arrowstyle="-|>", color="#27ae60", lw=1.8))

# Labels
ax.text(7.0, 4.8, "INDEXING (offline — done once)", fontsize=10,
        color='#2c3e50', fontweight='bold', ha='center')
ax.text(7.0, 0.5, "QUERYING (online — per request)", fontsize=10,
        color='#2c3e50', fontweight='bold', ha='center')

plt.title("RAG Architecture — Two-Phase Pipeline", fontsize=13, fontweight='bold', pad=15)
plt.tight_layout()
plt.savefig("images/rag_pipeline.png", dpi=130, bbox_inches='tight')
plt.show()

---
## Model Types for RAG

RAG has two distinct jobs that need different model architectures:

| Job | Model type | Direction | Examples | Why |
|-----|-----------|-----------|---------|-----|
| **Embedding** (indexing + retrieval) | Auto-encoding | Bidirectional — sees full context | BERT, `text-embedding-3-small` | Needs full sentence context to compute a good vector |
| **Generation** (answer production) | Auto-regressive | Left-to-right only | GPT-4o, LLaMA, Claude | Predicts next token from all previous tokens |

**Key insight:** You use two different models in RAG — an encoder to embed documents/queries, and a decoder to generate the final answer. They never share weights.

```
Embedding model:  "The cat sat on the mat" → [0.23, -0.45, 0.67, ...]  (768-dim vector)
Generation model: "The cat sat on" → "the mat"  (next-token prediction)
```

---
## Vector Embeddings & Stores

An embedding converts text to a dense float vector such that **semantic similarity = geometric closeness**.

```
"What is photosynthesis?"  → [0.12, -0.34, 0.89, ...]   ┐
"How do plants make food?" → [0.11, -0.33, 0.87, ...]   ┘  cosine similarity ≈ 0.98  ← same topic
"Stock market crash 1929"  → [0.78,  0.42, -0.23, ...]      cosine similarity ≈ 0.12  ← different topic
```

**Popular embedding models:**

| Model | Dims | Context | Notes |
|-------|------|---------|-------|
| `text-embedding-3-small` | 1536 | 8K tokens | Best price/quality for RAG (OpenAI) |
| `text-embedding-3-large` | 3072 | 8K tokens | Higher quality, 5× cost |
| `BAAI/bge-small-en-v1.5` | 384 | 512 tokens | Free, runs locally, surprisingly good |
| `nomic-embed-text` | 768 | 8K tokens | Free, local, long context |

**Popular vector stores:**

| Store | Best for | Notes |
|-------|---------|-------|
| **ChromaDB** | Local dev, notebooks | Embedded (no server needed), persists to disk |
| **FAISS** | High-performance CPU search | Facebook, in-memory, no persistence |
| **Pinecone** | Production cloud | Managed, scales automatically |
| **Weaviate** | Hybrid search | Keyword + vector in one query |

---
## Document Loading

First step: get your documents into Python. Two common formats — plain text and PDF.

In [ ]:
import os
import glob
from pathlib import Path


def load_text_files(folder: str, pattern: str = "**/*.txt") -> list[dict]:
    """Load all text files from a folder recursively."""
    documents = []
    for file_path in glob.glob(os.path.join(folder, pattern), recursive=True):
        with open(file_path, "r", encoding="utf-8", errors="ignore") as f:
            content = f.read()
        documents.append({
            "content": content,
            "source": file_path,
            "filename": Path(file_path).name,
            "char_count": len(content),
        })
    print(f"Loaded {len(documents)} files")
    return documents


def load_pdf(file_path: str) -> dict:
    """Load a PDF and concatenate all page text."""
    from pypdf import PdfReader          # pip install pypdf
    reader = PdfReader(file_path)
    text = "\n".join(page.extract_text() or "" for page in reader.pages)
    return {
        "content": text,
        "source": file_path,
        "filename": Path(file_path).name,
        "pages": len(reader.pages),
        "char_count": len(text),
    }


# ── Example usage ─────────────────────────────────────────────────────────
# docs = load_text_files("./my_docs")
# pdf  = load_pdf("./report.pdf")
# print(f"PDF: {pdf['pages']} pages, {pdf['char_count']:,} chars")

---
## Text Chunking

LLMs have a context window limit and vector search works best on short, focused passages. We split documents into overlapping chunks so that:
- Each chunk fits in the context window
- Sentences split across a boundary still appear in at least one chunk (via overlap)

```
Document:  [─────────────────────────────────────────────────────────]
Chunk 1:   [─────────────────────────────]
Chunk 2:                     [─────────────────────────────]
                              ↑──── overlap ────↑
Chunk 3:                                    [─────────────────────────────]
```

**Rule of thumb:** chunk_size=1000 chars, overlap=200 chars (~20%). Adjust based on your model's context window and average passage length.

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np


def chunk_documents(
    documents: list[dict],
    chunk_size: int = 1000,
    overlap: int = 200,
) -> list[dict]:
    """Split documents into overlapping chunks."""
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=overlap,
        # Try separators in order: paragraph → line → sentence → word → char
        separators=["\n\n", "\n", ". ", " ", ""],
    )
    chunks = []
    for doc in documents:
        for i, text in enumerate(splitter.split_text(doc["content"])):
            chunks.append({
                "content": text,
                "source": doc["source"],
                "chunk_id": i,
                "char_count": len(text),
            })
    return chunks


# ── Visualize chunk overlap on a synthetic document ───────────────────────
DOC_LEN    = 3000   # chars
CHUNK_SIZE = 1000
OVERLAP    = 200
STEP       = CHUNK_SIZE - OVERLAP   # 800

starts = list(range(0, DOC_LEN - OVERLAP, STEP))
while starts and starts[-1] + CHUNK_SIZE > DOC_LEN:
    starts[-1] = DOC_LEN - CHUNK_SIZE  # clamp last chunk
    if len(starts) > 1 and starts[-1] == starts[-2]:
        starts.pop()
        break

colors = ['#3498db', '#e74c3c', '#2ecc71', '#f39c12', '#9b59b6']

fig, ax = plt.subplots(figsize=(12, 3.5))
ax.set_xlim(-50, DOC_LEN + 50)
ax.set_ylim(-0.5, len(starts) + 0.3)
ax.axis('off')
fig.suptitle("Chunk Overlap Visualization", fontsize=12, fontweight='bold')

# Full document bar
ax.barh(len(starts), DOC_LEN, left=0, height=0.4,
        color='#ecf0f1', edgecolor='#bdc3c7', linewidth=1.5)
ax.text(DOC_LEN / 2, len(starts) + 0.15, f"Full document ({DOC_LEN:,} chars)",
        ha='center', va='bottom', fontsize=9, color='#7f8c8d')

# Chunk bars
for i, start in enumerate(starts):
    end = min(start + CHUNK_SIZE, DOC_LEN)
    ax.barh(i, end - start, left=start, height=0.55,
            color=colors[i % len(colors)], alpha=0.8, edgecolor='white', linewidth=1)
    ax.text(start + (end - start) / 2, i, f"Chunk {i+1}\n{end-start} chars",
            ha='center', va='center', fontsize=8, color='white', fontweight='bold')

    # Highlight overlap region with previous chunk
    if i > 0:
        prev_end = min(starts[i-1] + CHUNK_SIZE, DOC_LEN)
        overlap_len = prev_end - start
        if overlap_len > 0:
            ax.barh(i, overlap_len, left=start, height=0.55,
                    color='black', alpha=0.2, edgecolor='none')

ax.set_yticks(range(len(starts) + 1))
ax.set_yticklabels([f"Chunk {i+1}" for i in range(len(starts))] + ["Document"])
ax.yaxis.set_visible(True)
overlap_patch = mpatches.Patch(color='black', alpha=0.3, label=f'Overlap region ({OVERLAP} chars)')
ax.legend(handles=[overlap_patch], loc='lower right', fontsize=8)

plt.tight_layout()
plt.savefig("images/rag_chunking.png", dpi=120, bbox_inches='tight')
plt.show()
print(f"chunk_size={CHUNK_SIZE}, overlap={OVERLAP} → {len(starts)} chunks from {DOC_LEN}-char doc")

---
## Embeddings & ChromaDB

Embed every chunk into a vector and persist in ChromaDB. This is the **indexing phase** — done once, reused for all queries.

```
Chunk text  ─►  embedding model  ─►  [1536-dim float vector]  ─►  ChromaDB
```

ChromaDB stores: the vector, the raw text, and any metadata (source file, chunk index).

In [ ]:
from langchain_openai import OpenAIEmbeddings
from langchain_chroma import Chroma
from langchain_core.documents import Document

# OpenAI embedding model — 1536 dims, 8K token context
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")


def build_vector_store(
    chunks: list[dict],
    persist_directory: str = "./chroma_db",
) -> Chroma:
    """Embed all chunks and persist to ChromaDB."""
    docs = [
        Document(
            page_content=chunk["content"],
            metadata={"source": chunk["source"], "chunk_id": chunk["chunk_id"]},
        )
        for chunk in chunks
    ]
    vector_store = Chroma.from_documents(
        documents=docs,
        embedding=embeddings,
        persist_directory=persist_directory,
    )
    print(f"Indexed {len(docs)} chunks → {persist_directory}")
    return vector_store


def load_vector_store(persist_directory: str = "./chroma_db") -> Chroma:
    """Load a previously persisted ChromaDB store (no re-embedding needed)."""
    return Chroma(
        persist_directory=persist_directory,
        embedding_function=embeddings,
    )


# ── Usage ──────────────────────────────────────────────────────────────────
# chunks = chunk_documents(docs)
# vs = build_vector_store(chunks)          # first run — embeds everything
# vs = load_vector_store()                 # subsequent runs — instant load

---
## Similarity Search

Two search strategies:

| Strategy | What it does | When to use |
|----------|-------------|------------|
| **`similarity_search`** | Returns the K most similar chunks by cosine distance | Simple, fast, deterministic |
| **MMR** (Max Marginal Relevance) | Balances relevance *and* diversity — avoids returning near-duplicate chunks | When top-K might contain repetitive passages |

**Cosine similarity** — the standard metric for comparing embedding vectors:
```
cos(A, B) = (A · B) / (|A| × |B|)   ∈ [-1, 1]

1.0  = identical meaning
0.9+ = very similar topic
0.5  = loosely related
0.0  = unrelated
```

In [ ]:
def search_documents(vector_store: Chroma, query: str, k: int = 4) -> list[Document]:
    """MMR search — diverse and relevant."""
    return vector_store.max_marginal_relevance_search(
        query=query,
        k=k,
        fetch_k=20,      # fetch 20 candidates, re-rank to k diverse ones
        lambda_mult=0.5, # 0.0 = max diversity, 1.0 = max relevance
    )


def search_with_scores(vector_store: Chroma, query: str, k: int = 4) -> None:
    """Cosine similarity search with scores printed."""
    results = vector_store.similarity_search_with_relevance_scores(query, k=k)
    for doc, score in results:
        bar = "█" * int(score * 20)
        print(f"  {score:.3f} {bar:<20} {doc.metadata.get('source', '?')}")
        print(f"         {doc.page_content[:120].strip()}...\n")


# ── Usage ──────────────────────────────────────────────────────────────────
# search_with_scores(vs, "What is the refund policy?")
# chunks = search_documents(vs, "What is the refund policy?")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# ── Simulated cosine similarity scores for a query vs 8 document chunks ───
query = "How do I return a product?"

chunks = [
    "Our refund and return policy allows returns within 30 days.",
    "To return an item, visit the returns portal and print a label.",
    "Damaged goods must be reported within 48 hours of delivery.",
    "Shipping fees are non-refundable after dispatch.",
    "The company was founded in 1998 by two engineers.",
    "Our CEO spoke at the annual technology summit.",
    "New summer collection now available in stores.",
    "Contact customer service at support@example.com.",
]

# Simulated cosine similarity scores (as if from an embedding model)
scores = [0.91, 0.87, 0.72, 0.61, 0.18, 0.12, 0.09, 0.45]

# Sort by score
pairs = sorted(zip(scores, chunks), reverse=True)
scores_sorted = [p[0] for p in pairs]
labels_sorted = [p[1][:50] + "…" for p in pairs]

fig, ax = plt.subplots(figsize=(11, 4.5))
colors = ['#27ae60' if s > 0.7 else '#f39c12' if s > 0.4 else '#e74c3c'
          for s in scores_sorted]
bars = ax.barh(range(len(scores_sorted)), scores_sorted, color=colors, alpha=0.85,
               edgecolor='white', linewidth=0.8)

ax.set_yticks(range(len(labels_sorted)))
ax.set_yticklabels(labels_sorted, fontsize=8.5)
ax.set_xlabel("Cosine Similarity Score", fontsize=10)
ax.set_xlim(0, 1.05)
ax.axvline(0.6, color='gray', linestyle='--', alpha=0.6, label='Retrieval threshold (0.6)')
ax.set_title(f'Query: "{query}"\nRanked document chunks by similarity',
             fontsize=11, fontweight='bold')

# Score labels on bars
for bar, score in zip(bars, scores_sorted):
    ax.text(score + 0.01, bar.get_y() + bar.get_height() / 2,
            f"{score:.2f}", va='center', fontsize=8.5)

handles = [mpatches.Patch(color='#27ae60', label='High (>0.7) — retrieved'),
           mpatches.Patch(color='#f39c12', label='Medium (0.4–0.7)'),
           mpatches.Patch(color='#e74c3c', label='Low (<0.4) — filtered out')]
ax.legend(handles=handles, fontsize=8, loc='lower right')
plt.tight_layout()
plt.savefig("images/rag_similarity_scores.png", dpi=120, bbox_inches='tight')
plt.show()

---
## Full RAG Chain — LangChain LCEL

LCEL (LangChain Expression Language) lets you compose a pipeline with `|` operators. The chain runs left-to-right; each component's output is the next component's input.

```
{"context": retriever | format_docs, "question": passthrough}
    │                                     │
    ▼                                     ▼
 Top-K chunks joined            Original question string
    │                                     │
    └───────────────────┬─────────────────┘
                        ▼
                   prompt template
                        │
                        ▼
                    ChatOpenAI
                        │
                        ▼
                  StrOutputParser → final answer string
```

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough


RAG_PROMPT = ChatPromptTemplate.from_template("""You are an expert assistant. \
Answer the question using ONLY the provided context.
If the answer is not in the context, say "I don't have that information in the provided documents."
Do not speculate or use outside knowledge.

Context:
{context}

Question: {question}

Answer:""")


def build_rag_chain(vector_store: Chroma, model: str = "gpt-4o-mini"):
    """Build a complete RAG chain using LangChain LCEL."""
    retriever = vector_store.as_retriever(
        search_type="mmr",
        search_kwargs={"k": 4, "fetch_k": 20, "lambda_mult": 0.5},
    )
    llm = ChatOpenAI(model=model, temperature=0)

    def format_docs(docs: list[Document]) -> str:
        return "\n\n---\n\n".join(
            f"[Source: {d.metadata.get('source', '?')}]\n{d.page_content}"
            for d in docs
        )

    chain = (
        {"context": retriever | format_docs, "question": RunnablePassthrough()}
        | RAG_PROMPT
        | llm
        | StrOutputParser()
    )
    return chain


# ── Usage ──────────────────────────────────────────────────────────────────
# chain = build_rag_chain(vector_store)
# answer = chain.invoke("What is the company's refund policy?")
# print(answer)

---
## Keyword Retrieval — No Embeddings Needed

Sometimes you want fast, free, explainable retrieval without an embedding API call. BM25-style keyword overlap works surprisingly well for exact-match queries and is a useful fallback or baseline.

**When to prefer keyword over vector search:**
- Query uses exact product names / IDs (semantic search may drift)
- No budget for embedding API calls
- Need a fast sanity-check baseline
- Hybrid search: combine keyword pre-filter + vector re-rank

In [ ]:
import re
from collections import Counter


def keyword_retriever(
    documents: list[dict],
    query: str,
    top_k: int = 3,
    stopwords: set | None = None,
) -> list[tuple[float, dict]]:
    """BM25-inspired keyword overlap retrieval.
    
    Returns list of (score, doc) sorted by descending score.
    Score = |query_words ∩ doc_words| / |query_words|   (recall-like)
    """
    if stopwords is None:
        stopwords = {"the", "a", "an", "is", "in", "of", "to", "and", "or", "for", "with"}

    query_words = set(re.findall(r'\w+', query.lower())) - stopwords

    scored = []
    for doc in documents:
        doc_words = set(re.findall(r'\w+', doc["content"].lower())) - stopwords
        overlap = len(query_words & doc_words)
        if overlap > 0:
            score = overlap / max(len(query_words), 1)
            scored.append((score, doc))

    scored.sort(reverse=True, key=lambda x: x[0])
    return scored[:top_k]


def rag_with_keywords(documents: list[dict], query: str, client) -> str:
    """Full RAG using keyword retrieval instead of embeddings."""
    results = keyword_retriever(documents, query)
    if not results:
        return "No relevant documents found."

    context = "\n\n".join(
        f"[Score: {score:.2f}]\n{doc['content'][:500]}"
        for score, doc in results
    )
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": f"Answer using only this context:\n\n{context}"},
            {"role": "user",   "content": query},
        ],
        temperature=0,
    )
    return response.choices[0].message.content


# ── Quick test (no API needed) ─────────────────────────────────────────────
sample_docs = [
    {"content": "Our return policy allows returns within 30 days of purchase.", "source": "policy.txt"},
    {"content": "Shipping takes 3-5 business days for standard delivery.", "source": "shipping.txt"},
    {"content": "The CEO announced record quarterly earnings last week.", "source": "news.txt"},
]
results = keyword_retriever(sample_docs, "How do I return a product?")
for score, doc in results:
    print(f"Score {score:.2f} | {doc['source']}: {doc['content'][:60]}...")

---
## Gradio Interface

Wrap the RAG chain in a minimal web UI that anyone can use without touching Python.

In [ ]:
import gradio as gr


def create_rag_app(chain, title: str = "Expert Knowledge Worker"):
    """Wrap a LangChain RAG chain in a Gradio web interface."""

    def answer_question(question: str, history: list) -> str:
        if not question.strip():
            return "Please enter a question."
        return chain.invoke(question)

    demo = gr.ChatInterface(
        fn=answer_question,
        title=f"📚 {title}",
        description="Ask questions about your documents. Answers are grounded in retrieved passages.",
        examples=[
            "What is the refund policy?",
            "How long does shipping take?",
            "Who do I contact for support?",
        ],
        theme=gr.themes.Soft(),
    )
    return demo


# ── Launch ─────────────────────────────────────────────────────────────────
# demo = create_rag_app(chain)
# demo.launch()                           # opens browser tab
# demo.launch(share=True)                 # public URL via Gradio tunnel

---
## RAG Evaluation

How do you know if your RAG pipeline is working? Use an LLM as a judge to score each answer.

**Three dimensions to evaluate:**
| Metric | Question | What "bad" looks like |
|--------|---------|----------------------|
| **Faithfulness** | Is the answer supported by the retrieved context? | Answer contains facts not in the chunks |
| **Relevance** | Does the answer actually address the question? | Off-topic or evasive response |
| **Completeness** | Does the answer cover all parts of the question? | Partial answer, key detail missing |

In [ ]:
from openai import OpenAI


JUDGE_PROMPT = """You are evaluating a RAG system answer. Score on three dimensions (1-5):

Question: {question}
Retrieved context: {context}
Generated answer: {answer}
Expected answer (reference): {expected}

Rate each:
- Faithfulness (1-5): Is the answer fully supported by the context? (5=perfectly grounded, 1=hallucinated)
- Relevance (1-5): Does the answer address the question? (5=directly answers, 1=off-topic)
- Completeness (1-5): Does the answer cover all key points? (5=complete, 1=missing major details)

Respond as JSON: {{"faithfulness": X, "relevance": X, "completeness": X, "comment": "..."}}"""


def evaluate_rag(
    chain,
    vector_store: Chroma,
    test_cases: list[dict],
    judge_model: str = "gpt-4o-mini",
) -> list[dict]:
    """Evaluate RAG quality using LLM-as-judge.
    
    test_cases: [{"question": "...", "expected_answer": "..."}]
    Returns: list of result dicts with scores and generated answers.
    """
    import json
    judge = OpenAI()
    results = []

    for case in test_cases:
        q = case["question"]
        expected = case.get("expected_answer", "")

        # Get retrieved context
        retrieved = vector_store.similarity_search(q, k=4)
        context = "\n\n".join(d.page_content for d in retrieved)

        # Generate answer
        generated = chain.invoke(q)

        # Judge
        eval_resp = judge.chat.completions.create(
            model=judge_model,
            messages=[{
                "role": "user",
                "content": JUDGE_PROMPT.format(
                    question=q, context=context[:1000],
                    answer=generated, expected=expected,
                ),
            }],
            temperature=0,
        )
        try:
            scores = json.loads(eval_resp.choices[0].message.content)
        except json.JSONDecodeError:
            scores = {"faithfulness": 0, "relevance": 0, "completeness": 0,
                      "comment": "parse error"}

        results.append({
            "question": q,
            "generated": generated,
            "expected": expected,
            **scores,
        })

    # Summary
    if results:
        avg_f = sum(r["faithfulness"] for r in results) / len(results)
        avg_r = sum(r["relevance"]    for r in results) / len(results)
        avg_c = sum(r["completeness"] for r in results) / len(results)
        print(f"Avg Faithfulness: {avg_f:.1f}/5  |  "
              f"Relevance: {avg_r:.1f}/5  |  "
              f"Completeness: {avg_c:.1f}/5  ({len(results)} questions)")

    return results


# ── Usage ──────────────────────────────────────────────────────────────────
# test_cases = [
#     {"question": "What is the return window?", "expected_answer": "30 days"},
#     {"question": "How do I contact support?",  "expected_answer": "support@example.com"},
# ]
# results = evaluate_rag(chain, vector_store, test_cases)

---
## Summary

### Full pipeline at a glance

```python
# 1. Load
docs = load_text_files("./my_docs")

# 2. Chunk
chunks = chunk_documents(docs, chunk_size=1000, overlap=200)

# 3. Index (once)
vector_store = build_vector_store(chunks)
# or: vector_store = load_vector_store()   # subsequent runs

# 4. Build chain
chain = build_rag_chain(vector_store)

# 5. Query
answer = chain.invoke("What is the return policy?")
```

### Concept map

| Concept | What it is | Key decision |
|---------|-----------|--------------|
| **Embedding model** | Auto-encoder that converts text → vector | `text-embedding-3-small` is best balance for most RAG |
| **Chunk size** | Max chars per passage fed to the LLM | 500–1500 chars; overlap ~20% |
| **Vector store** | DB storing embeddings + metadata | ChromaDB (local), Pinecone (cloud) |
| **MMR search** | Diversity-aware retrieval | Prefer over pure cosine when docs are repetitive |
| **LCEL chain** | `retriever \| prompt \| llm \| parser` | Swap any component without rewriting the rest |
| **LLM-as-judge** | GPT-4 scores faithfulness / relevance / completeness | Run on a held-out test set before shipping |
| **Keyword retrieval** | Word overlap, no embeddings | Use as baseline or hybrid pre-filter |

### Common failure modes

| Symptom | Likely cause | Fix |
|---------|-------------|-----|
| Answer says "I don't know" but info is in docs | Retrieval miss — wrong chunk selected | Lower `lambda_mult`, increase `k`, revisit chunk size |
| Answer contains made-up facts | Hallucination — context too thin | Increase `k`, check embedding model quality |
| Slow first query | Embedding model cold start | Pre-warm with a dummy query |
| Very similar chunks returned | Duplicate passages in docs | Use MMR instead of similarity_search |